# Tutorial 4. Custom and library algorithms

Use five JNCC and Cefas seabed images from the 2013 East of Haig Fras survey to learn two ways to extend a pipeline. This is a separate collection from the 2012 AUV imagery in Tutorials 1 and 2. Run each cell in order with **Shift+Enter**. Use the **PAIDIVER** kernel with Paidiverpy 0.4.1 and Library Layer support.

| Step | Why we apply it |
| --- | --- |
| Open the five JPEGs | Keeping the exercise small makes it quick to repeat. The iFDO file pairs each JPEG with a timestamp, coordinates, and its source URL. |
| Grayscale | Convert to one intensity channel so it is easy to see what the two filters do and so the inversion and median filter operate on brightness alone. |
| Custom intensity inversion | Run code from the workshop’s `custom_algorithm/invert.py` to make an obvious visual change. This is a demonstration for learning how to connect your own code. |
| OpenCV median filter | Replace each intensity with the median of a 5 × 5 neighbourhood. Compare the result with the inverted image to see how the filter changes fine texture. |

The source metadata credits JNCC and Cefas and records the Open Government Licence v3.0. See the [source collection](https://jncc.resourcespace.com/pages/search.php?search=%21collection99) and [survey report](https://data.jncc.gov.uk/data/a74dc2aa-e8e9-4f23-81a2-287eefa7a47d/JNCC-CEFAS-28-FINAL-WEB.pdf).

## 1. Open the `benthic_ifdo2` sample and create the pipeline

The workshop uses five images from the Paidiverpy sample `benthic_ifdo2`. The cell reads the subset iFDO file, copies matching images from your local cache, and downloads any missing images using the source URLs. It saves them under `tutorials/images/benthic_ifdo2` for later runs. You need internet access for files that are absent from the cache.

The final lines tell Paidiverpy where to find the files and add grayscale as the first processing step. The output shows the configured steps. Paidiverpy may warn about missing fields such as depth and camera attitude in the source metadata. The image filters in this exercise use the pixels and do not depend on those fields. Later cells add inversion and median filtering before running the pipeline.

In [ ]:
import json
from pathlib import Path
from shutil import copyfile
from tempfile import TemporaryDirectory

import numpy as np
import requests
from PIL import Image
from paidiverpy.config import Configuration
from paidiverpy.colour_layer import ColourLayer
from paidiverpy.custom_layer import CustomLayer
from paidiverpy.library_layer import LibraryLayer
from paidiverpy.pipeline import Pipeline

tutorial_dir = Path.cwd().resolve()
metadata_name = "metadata_jncc_workshop_ifdo.json"
if not (tutorial_dir / "metadata" / metadata_name).is_file():
    tutorial_dir = tutorial_dir / "tutorials"
metadata_path = tutorial_dir / "metadata" / metadata_name
if not metadata_path.is_file():
    msg = "Open Jupyter in the workshop checkout or its tutorials folder."
    raise FileNotFoundError(msg)

image_dir = tutorial_dir / "images" / "benthic_ifdo2"
image_dir.mkdir(parents=True, exist_ok=True)
cache_root = Path.home() / ".paidiverpy_cache"
cache_dir = cache_root / "benthic_ifdo2" / "images"
if not cache_dir.is_dir():
    cache_dir = cache_root / "jncc" / "images"  # Reuse the earlier local cache if present.
items = json.loads(metadata_path.read_text(encoding="utf-8"))["image-set-items"]

with TemporaryDirectory(dir=image_dir) as staging, requests.Session() as session:
    for filename, metadata in items.items():
        destination = image_dir / filename
        if destination.is_file():
            continue
        temporary = Path(staging) / filename
        cached_image = cache_dir / filename
        if cached_image.is_file():
            copyfile(cached_image, temporary)
        else:
            with session.get(metadata["image-handle"], stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        stream.write(chunk)
        with Image.open(temporary) as image:
            image.verify()
        temporary.replace(destination)

config = Configuration(add_general={
    "input_path": image_dir,
    "metadata_path": metadata_path,
    "metadata_type": "IFDO",
    "image_open_args": "JPG",
    "output_path": tutorial_dir / "output" / "extensions",
    "n_jobs": 1,
})
pipeline = Pipeline(config=config, raise_error=True)
pipeline.add_step("grayscale", ColourLayer, {"mode": "grayscale"})
pipeline

## 2. Add a custom Python step

A custom step loads a class from a Python file. Open `custom_algorithm/invert.py` in Jupyter’s file browser. In the `invert` method, each pixel is subtracted from the largest value that its integer data type can represent. For an 8-bit image, that means 0 becomes 255 and 255 becomes 0.

The name `invert` identifies the method to call, and `class_name` identifies the class containing it. Use `processing_type: image` to apply the method to each image separately.

In [ ]:
pipeline.add_step(
    "invert",
    CustomLayer,
    {
        "file_path": str(tutorial_dir / "custom_algorithm" / "invert.py"),
        "class_name": "InvertColour",
        "processing_type": "image",
    },
)

## 3. Add an OpenCV function and run

The Library Layer calls a function already supplied by OpenCV or scikit-image. Give Paidiverpy the library, the function name, and its arguments. Paidiverpy supplies the current image automatically. Here, `ksize: 5` sets the median filter window width and height. Start with the supplied value; to experiment, try the odd value 3.

Run the pipeline once and expand the image gallery. The same source image appears at each stage.

In [ ]:
pipeline.add_step(
    "median_filter",
    LibraryLayer,
    {"library": "opencv", "function": "medianBlur", "params": {"ksize": 5}},
)
pipeline.run()
pipeline.images

## 4. Check how each algorithm changed the result

Use the thumbnail gallery to see the large change from the inversion and the subtler effect of the median filter. The table below measures the difference between consecutive stages on the same image. The mean absolute difference is the average change in intensity per pixel. Values greater than zero confirm the pixels changed.

Change `image_index` to inspect another image. To try a different median window, edit it in the previous cell, then rerun from Step 1 to create a fresh pipeline.

In [ ]:
import pandas as pd

image_index = 0
stage_indices = range(len(pipeline.images.steps) - 3, len(pipeline.images.steps))
arrays = []
for index in stage_indices:
    image = pipeline.images.get_step(step=index).isel(filename=image_index)
    height = int(image["original_height"].item())
    width = int(image["original_width"].item())
    pixels = image["images"].isel(y=slice(0, height), x=slice(0, width))
    arrays.append(pixels.to_numpy().astype(float))

before, inverted, filtered = arrays
pd.DataFrame({
    "Comparison": ["Before inversion → inverted", "Inverted → median filtered"],
    "Mean absolute intensity change": [
        np.abs(inverted - before).mean(),
        np.abs(filtered - inverted).mean(),
    ],
})